# Stage 2: Pseudo-Label Generation using Llama-3-8B

[1] Restoring environment:

In [ ]:
!pip install transformers datasets scikit-learn groq pandas numpy torch -q
print("Libraries ready.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/KD_Project/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/KD_Project/val.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/KD_Project/test.csv')
df       = pd.read_csv('/content/drive/MyDrive/KD_Project/full_dataset.csv')

print(f"Train : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")
print(f"Test  : {len(test_df)} samples")
print(f"Full  : {len(df)} samples")
print("Session restored successfully.")

In [ ]:
import os
from groq import Groq

GROQ_API_KEY = "gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # Replace with your actual API key
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
client = Groq(api_key=GROQ_API_KEY)

print("Groq client ready.")

In [ ]:
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None - change runtime!'}")

[2] Prompt Function:

In [ ]:
import json
import time

def get_pseudo_labels(sentence, client, retries=3):
    """
    Send one financial sentence to Llama-3.1-8B and get
    sentiment + urgency labels with confidence score.
    Returns a dict or None if all retries fail.
    """

    prompt = f"""You are a financial analyst. Classify the sentence below.

Sentence: "{sentence}"

Instructions:
1. Sentiment: choose exactly one of: positive, negative, neutral
2. Urgency: choose exactly one of: urgent, non-urgent
   - urgent = immediate action required, regulatory enforcement, crisis,
               major loss, legal action, market-moving announcement
   - non-urgent = general reporting, historical data, routine update
3. Confidence: your confidence in BOTH labels combined (0.00 to 1.00)

Respond ONLY with valid JSON. No explanation. No extra text.
Format: {{"sentiment": "...", "urgency": "...", "confidence": 0.XX}}"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=60,
                temperature=0.0    # deterministic output — critical for labelling consistency
            )

            raw = response.choices[0].message.content.strip()

            # Clean common JSON formatting issues from LLM output
            raw = raw.replace("```json", "").replace("```", "").strip()

            result = json.loads(raw)

            # Validate all required fields are present
            assert "sentiment"  in result
            assert "urgency"    in result
            assert "confidence" in result

            # Validate label values are within expected schema
            assert result["sentiment"] in ["positive", "negative", "neutral"]
            assert result["urgency"]   in ["urgent", "non-urgent"]
            assert 0.0 <= float(result["confidence"]) <= 1.0

            return result

        except (json.JSONDecodeError, AssertionError, KeyError):
            # Malformed JSON or wrong label — retry
            time.sleep(1)
            continue

        except Exception as e:
            # Rate limit or network error — wait longer before retry
            print(f"  API error on attempt {attempt+1}: {str(e)[:60]}")
            time.sleep(5)
            continue

    return None   # all retries exhausted


print("Prompt function defined successfully.")

Prompt function defined successfully.


[3] Small Batch Testing

In [ ]:
print("Testing on 5 samples...\n")
print(f"{'SENTENCE':<65} {'SENTIMENT':<12} {'URGENCY':<14} {'CONF'}")
print("-" * 105)

for i in range(5):
    sentence = train_df["text"].iloc[i]
    result   = get_pseudo_labels(sentence, client)

    if result:
        print(f"{sentence[:63]:<65} {result['sentiment']:<12} {result['urgency']:<14} {result['confidence']:.2f}")
    else:
        print(f"{sentence[:63]:<65} FAILED")

Testing on 5 samples...

SENTENCE                                                          SENTIMENT    URGENCY        CONF
---------------------------------------------------------------------------------------------------------
The major breweries increased their domestic beer sales by 4.5    positive     non-urgent     0.95
CapMan , an asset manager , has EUR 3bn worth of assets under m   neutral      non-urgent     0.95
`` The trend in the sports and leisure markets was favorable in   positive     non-urgent     0.95
`` In the newly formed company YIT Stavo the local contact netw   positive     non-urgent     0.95
`` We have a group of 120 volunteers made up of Digicel employe   positive     non-urgent     0.95


[4] Full Pseudo-Label Generation with Auto-Save

In [ ]:
import os

SAVE_PATH     = '/content/drive/MyDrive/KD_Project/pseudo_labels_progress.csv'
CONFIDENCE_THRESHOLD = 0.70

# Resume from checkpoint if a previous run was interrupted
if os.path.exists(SAVE_PATH):
    existing     = pd.read_csv(SAVE_PATH)
    start_idx    = len(existing)
    results_list = existing.to_dict('records')
    print(f"Resuming from checkpoint at index {start_idx}.")
else:
    start_idx    = 0
    results_list = []
    print("Starting fresh labelling run.")

total    = len(train_df)
failed   = 0
start_time = time.time()

print(f"Total samples to label : {total}")
print(f"Remaining              : {total - start_idx}")
print(f"Confidence threshold   : {CONFIDENCE_THRESHOLD}")
print(f"Saving every 100 samples to Drive.")
print("-" * 60)

for idx in range(start_idx, total):
    sentence      = train_df["text"].iloc[idx]
    gold_sentiment = train_df["sentiment"].iloc[idx]

    result = get_pseudo_labels(sentence, client)

    if result:
        results_list.append({
            "idx"             : idx,
            "text"            : sentence,
            "gold_sentiment"  : gold_sentiment,
            "pseudo_sentiment": result["sentiment"],
            "pseudo_urgency"  : result["urgency"],
            "confidence"      : float(result["confidence"]),
        })
    else:
        # Record failed samples — do not skip silently
        failed += 1
        results_list.append({
            "idx"             : idx,
            "text"            : sentence,
            "gold_sentiment"  : gold_sentiment,
            "pseudo_sentiment": "FAILED",
            "pseudo_urgency"  : "FAILED",
            "confidence"      : 0.0,
        })

    # Progress update every 100 samples
    if (idx + 1) % 100 == 0:
        elapsed   = time.time() - start_time
        remaining = (elapsed / (idx - start_idx + 1)) * (total - idx - 1)
        pct       = (idx + 1) / total * 100

        # Save checkpoint to Drive
        pd.DataFrame(results_list).to_csv(SAVE_PATH, index=False)

        print(f"[{idx+1:>4}/{total}] {pct:.1f}% complete | "
              f"Failed: {failed} | "
              f"Elapsed: {elapsed/60:.1f}m | "
              f"ETA: {remaining/60:.1f}m | "
              f"Checkpoint saved.")

    # Small delay to stay within Groq free tier rate limits
    time.sleep(5)

# Final save
pd.DataFrame(results_list).to_csv(SAVE_PATH, index=False)

total_time = time.time() - start_time
print(f"\nLabelling complete.")
print(f"Total samples processed : {len(results_list)}")
print(f"Failed samples          : {failed}")
print(f"Total time              : {total_time/60:.1f} minutes")

Resuming from checkpoint at index 3876.
Total samples to label : 3876
Remaining              : 0
Confidence threshold   : 0.7
Saving every 100 samples to Drive.
------------------------------------------------------------

Labelling complete.
Total samples processed : 3876
Failed samples          : 0
Total time              : 0.0 minutes


[5] Validating Pseudo-Label Quality:

In [ ]:
from sklearn.metrics import cohen_kappa_score, classification_report

# Load final results
pseudo_df = pd.read_csv(SAVE_PATH)

# Separate clean vs failed samples
failed_df = pseudo_df[pseudo_df["pseudo_sentiment"] == "FAILED"]
clean_df  = pseudo_df[pseudo_df["pseudo_sentiment"] != "FAILED"].copy()

print(f"Total labelled    : {len(pseudo_df)}")
print(f"Clean labels      : {len(clean_df)} ({len(clean_df)/len(pseudo_df)*100:.1f}%)")
print(f"Failed labels     : {len(failed_df)} ({len(failed_df)/len(pseudo_df)*100:.1f}%)")

# Cohen's Kappa — teacher vs gold labels
kappa = cohen_kappa_score(
    clean_df["gold_sentiment"],
    clean_df["pseudo_sentiment"]
)

# Accuracy
accuracy = (clean_df["gold_sentiment"] == clean_df["pseudo_sentiment"]).mean()

print(f"\n{'='*45}")
print(f"PSEUDO-LABEL QUALITY METRICS")
print(f"{'='*45}")
print(f"Cohen's Kappa (κ) : {kappa:.4f}  (target: ≥ 0.75)")
print(f"Label Accuracy    : {accuracy:.4f}  (target: ≥ 0.88)")
print(f"{'='*45}")

# Status
if kappa >= 0.75 and accuracy >= 0.88:
    print("STATUS: PASS — Teacher quality sufficient for distillation.")
elif kappa >= 0.60:
    print("STATUS: MARGINAL — Proceed with confidence filtering.")
else:
    print("STATUS: FAIL — Review prompt template before proceeding.")

print(f"\nClassification Report (Teacher vs Gold):")
print(classification_report(
    clean_df["gold_sentiment"],
    clean_df["pseudo_sentiment"],
    target_names=["negative", "neutral", "positive"]
))

Total labelled    : 3876
Clean labels      : 3876 (100.0%)
Failed labels     : 0 (0.0%)

PSEUDO-LABEL QUALITY METRICS
Cohen's Kappa (κ) : 0.6362  (target: ≥ 0.75)
Label Accuracy    : 0.8044  (target: ≥ 0.88)
STATUS: MARGINAL — Proceed with confidence filtering.

Classification Report (Teacher vs Gold):
              precision    recall  f1-score   support

    negative       0.83      0.86      0.85       483
     neutral       0.81      0.88      0.84      2303
    positive       0.77      0.62      0.69      1090

    accuracy                           0.80      3876
   macro avg       0.80      0.79      0.79      3876
weighted avg       0.80      0.80      0.80      3876



[6] Applying Confidence Filter + Saving Final Training Data:

In [ ]:
# Apply confidence threshold filter
filtered_df = clean_df[clean_df["confidence"] >= CONFIDENCE_THRESHOLD].copy()
removed     = len(clean_df) - len(filtered_df)

print(f"Samples before filtering : {len(clean_df)}")
print(f"Samples removed (<0.70)  : {removed}")
print(f"Samples after filtering  : {len(filtered_df)}")
print(f"Retention rate           : {len(filtered_df)/len(clean_df)*100:.1f}%")

# Urgency label distribution
print(f"\nUrgency label distribution:")
print(filtered_df["pseudo_urgency"].value_counts().to_string())

print(f"\nSentiment label distribution (post-filter):")
print(filtered_df["pseudo_sentiment"].value_counts().to_string())

# Save final training-ready dataset
FINAL_PATH = '/content/drive/MyDrive/KD_Project/pseudo_labels_final.csv'
filtered_df.to_csv(FINAL_PATH, index=False)

print(f"\nFinal pseudo-labelled dataset saved to Drive.")
print(f"Path: {FINAL_PATH}")

Samples before filtering : 3876
Samples removed (<0.70)  : 0
Samples after filtering  : 3876
Retention rate           : 100.0%

Urgency label distribution:
pseudo_urgency
non-urgent    3747
urgent         129

Sentiment label distribution (post-filter):
pseudo_sentiment
neutral     2495
positive     881
negative     500

Final pseudo-labelled dataset saved to Drive.
Path: /content/drive/MyDrive/KD_Project/pseudo_labels_final.csv
